# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR<sup>2</sup> dataset using the `mlcroissant` library, following best practices for reproducible, structured data science. All data entities are referenced by their `@id` fields as defined in the dataset's Croissant schema.

### Dataset Source

- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Not dict-like, do not subscript or iterate
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and their IDs with their Croissant `@id` references.

We'll inspect the available record sets, fields, and columns using the dataset's schema.

In [ ]:
# List all available record sets and their corresponding @id values

record_sets = dataset.record_sets  # Attribute listing all record set metadata
print("Available RecordSets (@id, name):")
for rs in record_sets:
    print(f"- {rs.id} | {getattr(rs, 'name', '')}")
    # List associated fields
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.id} | {getattr(field, 'name', '')} | dataType: {getattr(field, 'data_type', None)}")
    print()

# Choose one record set to preview some records
if len(record_sets) > 0:
    main_record_set_id = record_sets[0].id
    print(f"\nSample records from RecordSet '{main_record_set_id}':")
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        if i > 2:  # Show only first 3 records
            break
        print(record)

## 3. Data Extraction
Load data from each record set into DataFrames for analysis. All entities (record sets, fields, columns) are referenced by their `@id` as per the Croissant schema definition.

In [ ]:
# Extract all tabular record sets into DataFrames for easier exploration
# Use the `@id` of each record set (as discovered in the previous section)
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Preview columns of the first record set
if len(record_set_ids) > 0:
    preview_id = record_set_ids[0]
    print(f"Columns in RecordSet '{preview_id}': {dataframes[preview_id].columns.tolist()}")
    dataframes[preview_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply typical data processing steps, such as:
- Filtering records based on a threshold
- Normalizing a numeric field
- Grouping by a categorical variable

**All fields are referenced by their `@id`.**


In [ ]:
# Select a DataFrame and one numeric field for analysis by their @id
main_rs = None
numeric_field_id = None
group_field_id = None

# Example: heuristically pick a likely numeric field (e.g. one with 'Age' or 'Interval' in name or float/int data)
# Use the first record set and try to auto-pick a numeric column
if len(record_set_ids) > 0:
    main_rs = record_set_ids[0]
    df = dataframes[main_rs]
    # Try to guess numeric columns
    numeric_candidates = [col for col in df.columns if df[col].dtype.kind in {'i', 'u', 'f'} or 'age' in col.lower() or 'interval' in col.lower()]
    if len(numeric_candidates) > 0:
        numeric_field_id = numeric_candidates[0]
    else:
        # fallback: pick first column
        numeric_field_id = df.columns[0]

    # Attempt to select a group field (categorical variable)
    group_candidates = [col for col in df.columns if 'sex' in col.lower() or 'group' in col.lower() or 'type' in col.lower() or df[col].dtype == object]
    if len(group_candidates) > 0:
        group_field_id = group_candidates[0]

    print(f"Using numeric field '@id': {numeric_field_id}")
    print(f"Using group field '@id': {group_field_id}")

    # Ensure the numeric field is converted properly
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Filter records based on a threshold (e.g., > mean of numeric field)
    if df[numeric_field_id].notnull().any():
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
        print(filtered_df[[numeric_field_id]].head())

        # Add normalized column
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # If a group field is present, group by it and show the group means for the numeric field
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by '{group_field_id}':")
            print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the main numeric field and relationship with the group field if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the numeric field
if main_rs and numeric_field_id and main_rs in dataframes:
    df = dataframes[main_rs]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group field available, plot mean numeric field value per group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated loading the FAIR<sup>2</sup> clinical oncology dataset using the `mlcroissant` library, explored its structure by referencing all entities by their `@id`, performed basic exploratory data analysis including filtering, normalization, and grouping, and visualized one of the key numeric fields. 

This approach can be extended to more advanced analytics and model development while preserving clear provenance via Croissant-based schema referencing.